In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np
from data_processing_pipeline import runProcessingPipeline
import joblib
import os
import json
configs = json.load(open("config.json"))
processing_configs = json.load(open("src/processing_config.json"))

In [2]:
input_df = pd.read_csv(os.path.join(configs['row_write_folder_path'], configs['all_row_data_key']))
input_df.head()

,State,District,Market,Commodity,Variety,Grade,Arrival_Date,Min_Price,Max_Price,Modal_Price,Commodity_Code
0,Andhra Pradesh,Anantapur,Anantapur,Ground Nut Seed,Ground Nut Seed,Medium,2011-01-01,3800.0,3900.0,3850.0,268
1,Andhra Pradesh,Anantapur,Dharmavaram,Groundnut,Local,Medium,2011-01-01,2700.0,2800.0,2750.0,10
2,Andhra Pradesh,Anantapur,Guntakal,Bengal Gram (Gram)(Whole),Gulabi,Medium,2011-01-01,2400.0,2600.0,2500.0,6
3,Andhra Pradesh,Anantapur,Guntakal,Ground Nut Seed,Ground Nut Seed,Medium,2011-01-01,4300.0,4500.0,4400.0,268
4,Andhra Pradesh,Anantapur,Hindupur,Dry Chillies,1st Sort,Medium,2011-01-01,4600.0,6800.0,5750.0,132


In [3]:
from src.s3_operations import S3BucketHandler

s3_handler = S3BucketHandler(
    bucket_name=configs['bucket_name']
)

processed_df = s3_handler.readS3Data(file_key='batch_processed_file_key', nrows=-1)
processed_df.head()

,State,District,Market,Commodity,Variety,Grade,Arrival_Date,Min_Price,Max_Price,Modal_Price,Commodity_Code
0,1,22,154,146,487,3,2011-01-01,0.000004,0.000004,0.000004,268
1,1,22,935,147,724,3,2011-01-01,0.000003,0.000003,0.000003,10
2,1,22,1212,36,498,3,2011-01-01,0.000003,0.000003,0.000003,6
3,1,22,1212,146,487,3,2011-01-01,0.000005,0.000005,0.000005,268
4,1,22,1285,111,12,3,2011-01-01,0.000005,0.000007,0.000006,132


In [5]:
len(input_df['Commodity'].unique())

369

In [52]:
import boto3

s3 = boto3.client('s3')

bucket = "market-price-data-vijay-takbhate"
prefix = "commodity_wise_data/"

response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)

files = []

if "Contents" in response:
    for obj in response["Contents"]:
        key = obj["Key"]
        if key != prefix:
            files.append(key.split("/")[-1].replace(".csv", "").lower())

print(files)

['absinthe', 'ajwan', 'alasande_gram', 'almond_(badam)', 'alsandikai', 'amaranthus', 'ambada_seed', 'amla_(nelli_kai)', 'amphophalus', 'antawala', 'anthorium', 'apple', 'apricot_(jardalu_or_khumani)', 'arecanut_(betelnut_or_supari)', 'arhar_(tur_or_red_gram)(whole)', 'arhar_dal_(tur_dal)', 'asalia', 'ashgourd', 'ashwagandha', 'asparagus', 'astera', 'avare_dal', 'bop', 'bajra_(pearl_millet_or_cumbu)', 'balekai', 'bamboo', 'banana', 'banana_-_green', 'barley_(jau)', 'bay_leaf_(tejpatta)', 'beans', 'beaten_rice', 'beetroot', 'bengal_gram_(gram)(whole)', 'bengal_gram_dal_(chana_dal)', 'ber_(zizyphus_or_borehannu)', 'betal_leaves', 'bhindi_(ladies_finger)', 'big_gram', 'binoula', 'bitter_gourd', 'black_gram_(urd_beans)(whole)', 'black_gram_dal_(urd_dal)', 'black_pepper', 'borehannu', 'bottle_gourd', 'brahmi', 'bran', 'brinjal', 'broken_rice', 'broomstick_(flower_broom)', 'bull', 'bullar', 'bunch_beans', 'butter', 'cabbage', 'calf', 'cane', 'capsicum', 'cardamoms', 'carnation', 'carrot', 'ca

In [53]:
total_commodities = list(input_df["Commodity"].unique())
s3_commodities = files

In [59]:
not_in_s3_cmdts = []
for commodity in total_commodities:
    if commodity.lower().replace(" ", "_").replace("/", "_or_") in s3_commodities:
        continue
    not_in_s3_cmdts.append(commodity)

In [60]:
len(not_in_s3_cmdts)

15

In [61]:
len(total_commodities) - len(s3_commodities)

16

['Same/Savi',
 'Ambady/Mesta',
 'Amranthas Red',
 'Muleti',
 'Ratanjot',
 'stone pulverizer',
 'Palash flowers',
 'Gudmar',
 'stevia',
 'vadang',
 'Calendula',
 'Bhui Amlaya',
 'Bael',
 'Kalmegh',
 'liquor turmeric']

In [81]:
for commodity, label in zip(input_df['Commodity'].unique(), processed_df['Commodity'].unique()):
    if commodity in not_in_s3_cmdts:
        save_path = os.path.join(prefix, commodity.replace("/", "_or_").replace(" ", "_"))
        print("Saving {}".format(save_path))
        # s3_handler.uploadToS3(file_key = save_path)

Saving commodity_wise_data/Same_or_Savi


(354, 369)

((30032282, 11), (30032282, 11))